# CS781 Capstone — Phase 1: legal BPE + two A3 decoders trained in parallel on 2×T4

**Kaggle settings:** Accelerator **GPU T4 ×2**, Internet **on**, secret **`HF`** = your Hugging Face token.
Run with **Save Version → Save & Run All (Commit)** so the 12 h session isn't killed by browser idle time.

| Stage | Where | What |
|---|---|---|
| 0 | this kernel | session clock, config (**fill the `None`s in the config cell**), HF login |
| 1 | `%%writefile` cells | `a1_tokenizer.py` (legal BPE), `arch1/2/3.py` (**your A3 code, verbatim**), `fastattn.py` + `train_lm.py` (plumbing) |
| 2 | CPU | param counts + SDPA/KV-cache equivalence check against your original forward |
| 2b | GPU 0 ‖ GPU 1, ~20–25 min | **runtime probe**: your real configs on synthetic tokens → tok/s, memory, throttling, eval/generation time, 12 h projection (`PROBE_ONLY` stops here) |
| 3 | CPU, ~40–70 min | download corpus → train 32k legal BPE → build `train.bin` / `val.bin` (skipped when prebuilt data is attached; `DATA_ONLY` stops here) |
| 4 | GPU 0 ‖ GPU 1 | model A and model B train **in parallel**, then each writes its own `prediction.json` |
| 5 | CPU | `submission_A.zip`, `submission_B.zip`, loss curves |

The training deadline is computed from the session clock started in the first code cell, so data building, training and
generation always fit inside `SESSION_HOURS`. To save the session's data build for later runs, add this notebook's
output as a Kaggle Dataset and set `PREBUILT_DATA_DIR` next time (then all ~11 h go to training).

**LLM declaration (§17.5):** architecture code in `arch1/2/3.py` is yours, copied unchanged. Every model / optimiser /
schedule value is left `None` for you to choose. Claude wrote the plumbing: tokenizer optimisation, the data pipeline,
`fastattn.py` (fused SDPA + KV cache wrapped around your modules, checked equivalent to your forward), `train_lm.py` and this notebook.

## Stage 0 — session clock, login, configuration

In [ ]:
# session clock + GPU check (run first: the training deadline is measured from here)
import os, sys, json, time, math, glob, shutil, random, hashlib, subprocess
SESSION_START = time.time()

import numpy as np
import torch

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

N_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| CPUs:", os.cpu_count(), "| GPUs:", N_GPUS,
      [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

In [ ]:
# Hugging Face login from Kaggle Secrets (never paste the token into the notebook)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

In [ ]:
# configuration
SEED = 42

# ---------------- session budget (plumbing) ----------------
SESSION_HOURS = 12.0        # Kaggle hard limit
SAFETY_MIN = 20             # stop everything this long before the limit
GEN_RESERVE_MIN = 45        # reserved after training for generation + zipping

# ---------------- run mode (plumbing) ----------------
# Session 1 (Accelerator: None / CPU only): DATA_ONLY = True  -> tokenizer + train.bin/val.bin, then stops.
#   Save Version (commit), then attach that version's output (or a Dataset made from it) as an input.
# Session 2 (Accelerator: GPU T4 x2): DATA_ONLY = False, RUN_PROBE = False (cell "runtime probe") -> trains.
DATA_ONLY = False

# ---------------- paths ----------------
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
PREBUILT_DATA_DIR = "auto"  # "auto": use an attached /kaggle/input/**/legal_bpe.json + train.bin if present, else build.
                            # Or an explicit path, e.g. "/kaggle/input/<dataset>/cs781_phase1/pretrain_data_bpe"; None = always build


def _find_prebuilt(root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", "legal_bpe.json"), recursive=True))
    ok = [os.path.dirname(h) for h in hits
          if all(os.path.exists(os.path.join(os.path.dirname(h), f)) for f in ("train.bin", "val.bin", "manifest.json"))]
    return ok[0] if ok else None


if PREBUILT_DATA_DIR == "auto":
    PREBUILT_DATA_DIR = _find_prebuilt()
if PREBUILT_DATA_DIR:
    print("Prebuilt data found:", PREBUILT_DATA_DIR, "-> stage 3 (download, tokenizer, data build) is skipped")
else:
    print("No prebuilt data attached -> stage 3 will build the tokenizer + train.bin in this session (~40-70 min)")
assert DATA_ONLY or N_GPUS > 0, ("no GPU visible: set DATA_ONLY = True for the CPU-only data build, "
                                 "or set Accelerator to GPU T4 x2")
DATA_DIR = PREBUILT_DATA_DIR or os.path.join(OUTPUT_DIR, "pretrain_data_bpe")
TOKENIZER_PATH = os.path.join(DATA_DIR, "legal_bpe.json")
RESUME = {"A": None, "B": None}   # e.g. "/kaggle/input/<prev-output>/cs781_phase1/run_A/last.pt"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# ---------------- tokenizer (32k, your A1 byte-level BPE, legal pre-tokenisation) ----------------
VOCAB_SIZE = 32_000         # includes the 32 special tokens; multiple of 64
TOK_TRAIN_CHARS = 200_000_000
TOK_SOURCE_MIX = {"supreme_court": 0.40, "acts": 0.30, "high_courts": 0.15, "tribunals": 0.15}  # your best mix (sc_acts_heavy)
TOK_MIN_WORD_FREQ = 2       # speed knob: ignore whitespace words seen once when learning merges

# ---------------- pretraining data mixture (values from your earlier data pipeline) ----------------
TOTAL_TRAIN_TOKENS = 1_000_000_000
SC_MAX_EPOCHS, SC_MAX_SHARE = 4.0, 0.70
ACTS_MAX_EPOCHS, ACTS_MAX_SHARE = 2.0, 0.08
SC_VAL_PERCENT = 2
MIN_WORDS = 100
MAX_FERTILITY = 2.0
N_VAL_PROMPTS = 300
PROFILE_CHARS = 30_000_000  # per-source sample for the DSIR-style unigram profiles
HT_SCAN_FACTOR = 4          # score this many x the needed HC+Tribunal tokens (random sample), keep the most SC-like

# =====================================================================================
# YOUR CHOICES (course policy: model design + hyperparameters are yours). Fill every None.
# Target from our discussion: ~150M parameters, vocab 32k. The sanity-check cell prints the exact count.
# =====================================================================================
MODEL_A = {                          # architecture 1: decoder-only baseline, MHA (arch1 = attention1 + decoder1)
    "module": "arch1",
    "model": dict(vocab_size=VOCAB_SIZE, d_model=768, n_layers=17, n_heads=12, d_ff=3328,
                  max_seq_len=896, dropout=0.1, pos_emb_type="sinusoidal",
                  norm_first=True, attention_scale=0.125),
    "init_std": 0.02,                # None = PyTorch default init (note: nn.Embedding default is N(0, 1))
}
MODEL_B = {                          # architecture 2: efficient attention
    "module": "arch3",               # "arch3" (attention3 + rope + decoder3: GQA/MQA + RoPE) or "arch2" (MLA, sinusoidal)
    "model": dict(vocab_size=VOCAB_SIZE, d_model=768, n_layers=18, n_heads=12, n_kv_heads=4, d_ff=3456,
                  max_seq_len=896, dropout=0.1, pos_emb_type="rope", norm_first=True, attention_scale=0.125),
                  # for arch2 also add: attention_type="mla", kv_latent_dim=...
    "init_std": 0.02,
}
TRAIN = dict(
    batch_tokens=8192,               # global batch in tokens (micro-batch + grad-accum are derived to fit T4 memory)
    lr=3e-4, min_lr=3e-5, warmup_steps=1000, schedule="cosine",   # "cosine" | "linear" | "constant"
    weight_decay=0.1, betas=(0.9, 0.95), eps=1e-8, grad_clip=1.0,
    max_tokens=350_000_000,          # optional token cap; otherwise the schedule runs to the time deadline
    # plumbing
    eval_every_min=30, eval_batches=40, log_every=50, compile=True, fast_attention=True, seed=SEED,
)
TRAIN_A = dict(TRAIN)                # per-model overrides go here, e.g. TRAIN_B = dict(TRAIN, lr=...)
TRAIN_B = dict(TRAIN)

GEN = dict(                          # decoding (starter defaults; tuning them is yours)
    checkpoint="best",               # "best" (lowest val loss) or "last"
    target_words=128, max_new_tokens=256, batch_size=32,
    temperature=0.1, top_k=None, top_p=0.95, repetition_penalty=1.0,
    n_val_prompts=100,               # also continue held-out SC prompts -> val_generations.json for local scoring
    seed=SEED,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


def missing(d, prefix=""):
    out = []
    for k, v in d.items():
        if isinstance(v, dict):
            out += missing(v, f"{prefix}{k}.")
        elif v is None and k not in ("attention_scale", "init_std", "max_tokens", "top_k"):
            out.append(prefix + k)
    return out

todo = missing(MODEL_A, "MODEL_A.") + missing(MODEL_B, "MODEL_B.") + missing(TRAIN_A, "TRAIN_A.") + missing(TRAIN_B, "TRAIN_B.")
assert not todo, "Fill these in the configuration cell before running: " + ", ".join(todo)
print("Configuration loaded. Data dir:", DATA_DIR)

## Stage 1 — code files (written into `/kaggle/working`, nothing is sourced from outside the notebook)
`arch1/2/3.py` are your A3 files concatenated (the `from attention/rope import ...` lines are dropped because everything now lives in one module per architecture; two bug fixes are marked `# FIX (Claude)`).

In [ ]:
%%writefile a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile arch1.py
# Architecture 1 — attention1.py + decoder1.py (your code)
# ======== attention1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Multi-head causal self-attention, implemented from raw tensor ops."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Hook for Q1.4. None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj as nn.Linear
        # (no bias). All should map d_model -> d_model.
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # TODO: register a (max_seq_len, max_seq_len) lower-triangular boolean mask as a
        # buffer named `causal_mask` (True = allowed to attend, False = masked out), so
        # you don't rebuild it on every forward call.
        causal_mask = torch.tril(
            torch.ones(max_seq_len,max_seq_len,dtype=torch.bool)
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        # TODO 1: project to q, k, v  (each (B, T, C))
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # TODO 2: reshape each to (B, n_heads, T, d_head) — split C into (n_heads, d_head)
        #         and move n_heads next to the batch dim.
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        
        # TODO 3: compute attention scores = (q @ k^T) * self.attention_scale
        #         -> (B, n_heads, T, T). Do NOT hard-code 1/sqrt(d_head): Q1.4
        #         changes this hook for the scaling ablation.
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
        # TODO 4: apply the causal mask (set disallowed positions to -inf before softmax)
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
        # TODO 5: softmax over the last dim, apply self.attn_dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
        # TODO 6: weighted sum with v -> (B, n_heads, T, d_head)
        weighted_sum = attention_weights@v
        # TODO 7: reshape back to (B, T, C) and apply self.out_proj + self.resid_dropout
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          # == n_heads for plain MHA; used in Question 2
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # or "rope", used in Question 3
    attention_scale: float = None      # Q1.4 hook; None -> 1/sqrt(d_head)
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # cfg.norm_first = true for pre norm
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile arch2.py
# Architecture 2 option — attention2.py + decoder2.py (your code: GQA / MLA)
# ======== attention2.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA / MQA / GQA via n_kv_heads."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads   # how many query heads share each kv head
        self.d_head = d_model // n_heads
        # Preserve the Q1.4 hook in later questions. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: q_proj maps d_model -> n_heads * d_head (as before).
        # TODO: k_proj, v_proj map d_model -> n_kv_heads * d_head (SMALLER than before
        #       whenever n_kv_heads < n_heads -- this is where the parameter savings
        #       and (in a real serving system) the KV-cache savings come from).
        # FIX (Claude): `d_head` was undefined here (NameError); use self.d_head
        self.q_proj = nn.Linear(d_model,n_heads*self.d_head,bias=False)
        self.k_proj = nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj = nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        # TODO 1: project q -> (B, n_heads, T, d_head); k, v -> (B, n_kv_heads, T, d_head)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_kv_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_kv_heads,self.d_head)).transpose(1,2)
        # TODO 2: expand k and v from n_kv_heads to n_heads by repeating each kv head
        #         `self.n_rep` times (torch.repeat_interleave along the head dim is the
        #         natural tool here) so shapes line up for a standard batched matmul.

        k = k.repeat_interleave(self.n_rep,dim=1)
        v = v.repeat_interleave(self.n_rep,dim=1)

        # TODO 3: same attention as Q1, using (q @ k^T) * self.attention_scale,
        #         then causal mask + softmax + weighted sum.
        # TODO 4: reshape back to (B, T, C), apply out_proj.



        attention_scores = (q@k.transpose(2,3))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output
        
     
      


class MultiHeadLatentAttention(nn.Module):
    """Q2-only joint KV compression; use the equations in Q2.1."""
    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len,
                 dropout=0.0, attention_scale=None):
        super().__init__()
        # TODO: validate d_model % n_heads == 0 and 0 < kv_latent_dim < d_model.
        assert d_model%n_heads==0
        assert kv_latent_dim>0 and kv_latent_dim<d_model
        # Expose n_heads, d_head, kv_latent_dim, and attention_scale.
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.kv_latent_dim = kv_latent_dim
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)
        
        # None means 1/sqrt(d_head); preserve explicit attention_scale overrides.
        # All five projections must be bias-free nn.Linear modules with these names:
        self.q_proj = nn.Linear(d_model,d_model,bias=False)        # d_model -> d_model
        self.kv_down_proj = nn.Linear(d_model,kv_latent_dim,bias=False)  # d_model -> kv_latent_dim (shared by K and V)
        self.k_up_proj = nn.Linear(kv_latent_dim,d_model,bias=False)     # kv_latent_dim -> d_model
        self.v_up_proj = nn.Linear(kv_latent_dim,d_model,bias=False)     # kv_latent_dim -> d_model
        self.out_proj = nn.Linear(d_model,d_model,bias=False)      # d_model -> d_model
        # TODO: attention/residual dropout and a nonpersistent causal_mask buffer.
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        

    def forward(self, x):
        # TODO: compress once, reconstruct per-head K/V from the same latent,
        # compute causal scaled attention, concatenate heads, and project output.
        # Input/output shape: (B, T, d_model). Do not add positions in this module.
        B,T,d_model = x.shape
        q = self.q_proj(x)
        latent = self.kv_down_proj(x)
        k = self.k_up_proj(latent)
        v = self.v_up_proj(latent)
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,d_model))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder2.py (your A3 code; cross-file import removed, FIX lines marked) ========
# TODO 2.2: integrate n_kv_heads through the decoder architecture in this cell.
"""Decoder-only Transformer LM -- Block/DecoderLM updated to thread cfg.n_kv_heads
through to attention (everything else is unchanged from Question 1)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False
    attention_type: str = "standard"  # Q2 only: "standard" or "mla"
    kv_latent_dim: int = None


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE



class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        ) 

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass cfg.n_kv_heads as the new required argument to CausalSelfAttention
        if cfg.attention_type == "mla":
            if cfg.pos_emb_type != "sinusoidal":
                raise ValueError("MLA is restricted to Q2 with sinusoidal positions")
            self.attn = MultiHeadLatentAttention(
                cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len,
                cfg.dropout, attention_scale=cfg.attention_scale)
        elif cfg.attention_type == "standard":
            self.attn = CausalSelfAttention(
                cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                cfg.dropout, attention_scale=cfg.attention_scale)
        else:
            raise ValueError("attention_type must be 'standard' or 'mla'")
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output
      # unchanged from Question 1


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile fastattn.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10):
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0:
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile train_lm.py
# Plumbing: single-GPU training + prediction.json (launched once per GPU)
"""Train one hand-written A3 decoder on one GPU, then write its prediction.json (written with Claude).

Usage: CUDA_VISIBLE_DEVICES=0 python train_lm.py run_A.json

Every model/optimiser/schedule value comes from the JSON config filled in the notebook.
The script only guarantees the run finishes inside the Kaggle session: the LR schedule is
driven by wall-clock progress towards `deadline_unix`, so the decay always completes before
the deadline regardless of the measured throughput.

Probe mode (cfg["probe"] set, used by the Stage 2b runtime probe): same code path on synthetic
tokens, trains for a fixed window, then times worst-case generation with a stand-in tokenizer
instead of writing prediction.json.
"""
import os
import sys
import json
import math
import time
import queue
import random
import threading
import traceback
import gc
import importlib

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import fastattn
from a1_tokenizer import ByteBPE


def log(msg, **kv):
    rec = {"t": round(time.time(), 1), "msg": msg, **kv}
    print(json.dumps(rec), flush=True)
    with open(os.path.join(OUT, "log.jsonl"), "a") as f:
        f.write(json.dumps(rec) + "\n")


cfg = json.load(open(sys.argv[1]))
TAG, OUT = cfg["tag"], cfg["out_dir"]
os.makedirs(OUT, exist_ok=True)
TR, GEN = cfg["train"], cfg["gen"]
DEADLINE = cfg["deadline_unix"]
SESSION_END = cfg["session_end_unix"]

seed = TR["seed"]
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
CUDA = torch.cuda.is_available()         # CPU path exists only for local smoke tests
device = torch.device("cuda" if CUDA else "cpu")
AMP = torch.float16                      # T4 has no bf16 tensor cores
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---------------------------------------------------------------------------
# model
# ---------------------------------------------------------------------------
arch = importlib.import_module(cfg["module"])
mcfg = arch.LMConfig(**cfg["model"])
raw_model = arch.DecoderLM(mcfg)
if cfg.get("init_std"):
    std = cfg["init_std"]
    for m in raw_model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, std)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)
n_params = sum(p.numel() for p in raw_model.parameters())
if TR.get("fast_attention", True):
    fastattn.patch_model(raw_model)
raw_model.to(device)
T = mcfg.max_seq_len
V = mcfg.vocab_size
assert GEN["max_new_tokens"] < T, "gen.max_new_tokens must be < max_seq_len (checked before training, not after)"
log("model", tag=TAG, module=cfg["module"], params_M=round(n_params / 1e6, 2), seq_len=T)

# ---------------------------------------------------------------------------
# data: random windows from the uint16 token stream, prefetched on a thread
# ---------------------------------------------------------------------------
train_mm = np.memmap(os.path.join(cfg["data_dir"], "train.bin"), dtype=np.uint16, mode="r")
val_mm = np.memmap(os.path.join(cfg["data_dir"], "val.bin"), dtype=np.uint16, mode="r")
assert int(train_mm[:10_000_000].max()) < V, "token id >= vocab_size: data/tokenizer mismatch"


def windows(mm, starts):
    arr = np.stack([mm[s:s + T + 1] for s in starts]).astype(np.int64)
    t = torch.from_numpy(arr)
    t = t.pin_memory() if CUDA else t
    return t[:, :-1], t[:, 1:]


class Prefetcher:
    def __init__(self, mb, seed):
        self.mb, self.rng = mb, np.random.default_rng(seed)
        self.q = queue.Queue(maxsize=6)
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        hi = len(train_mm) - T - 1
        while True:
            self.q.put(windows(train_mm, self.rng.integers(0, hi, self.mb)))

    def next(self):
        x, y = self.q.get()
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)


def loss_fn(model, x, y):
    with torch.autocast("cuda", dtype=AMP, enabled=CUDA):
        logits = model(x)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), y.reshape(-1))


# ---------------------------------------------------------------------------
# micro-batch: largest divisor of the global batch that fits in memory next to AdamW state
# ---------------------------------------------------------------------------
batch_seqs = max(1, TR["batch_tokens"] // T)
free_mem = torch.cuda.mem_get_info()[0] + torch.cuda.memory_allocated() if CUDA else 0
opt_state_bytes = 2 * 4 * sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
budget = free_mem - opt_state_bytes - 1.5e9
candidates = sorted([d for d in range(1, min(batch_seqs, TR.get("max_micro_batch", 64)) + 1)
                     if batch_seqs % d == 0], reverse=True)
mb, peak = None, 0
if not CUDA:
    mb = min(candidates, key=lambda d: abs(d - 2))
for cand in candidates if CUDA else []:
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        x, y = windows(train_mm, np.arange(cand) * 997)
        raw_model.train()
        loss_fn(raw_model, x.to(device), y.to(device)).backward()
        raw_model.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated()
        if peak < budget:
            mb = cand
            break
    except torch.cuda.OutOfMemoryError:
        pass
    raw_model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
assert mb is not None, "even micro-batch 1 does not fit; reduce max_seq_len or model size"
accum = batch_seqs // mb
log("batch", global_seqs=batch_seqs, micro_batch=mb, grad_accum=accum, tokens_per_step=batch_seqs * T,
    probe_peak_GB=round(peak / 1e9, 2))

model = raw_model
if TR.get("compile", False):
    t_compile = time.time()
    try:
        model = torch.compile(raw_model)
        x, y = windows(train_mm, np.arange(mb) * 991)
        loss_fn(model, x.to(device), y.to(device)).backward()
        raw_model.zero_grad(set_to_none=True)
        log("torch.compile ok", seconds=round(time.time() - t_compile, 1))
    except Exception as e:                                   # fall back to eager
        model = raw_model
        raw_model.zero_grad(set_to_none=True)
        log("torch.compile failed, using eager", error=repr(e)[:300])

# ---------------------------------------------------------------------------
# optimiser (weight decay on >=2-D weights only)
# ---------------------------------------------------------------------------
decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() < 2]
groups = [{"params": decay, "weight_decay": TR["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}]
try:
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), eps=TR.get("eps", 1e-8), fused=CUDA)
except (RuntimeError, TypeError):
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), eps=TR.get("eps", 1e-8))
scaler = torch.amp.GradScaler("cuda", enabled=CUDA)

step, tokens_seen, train_seconds_before, best_val = 0, 0, 0.0, float("inf")
warmup_end_progress = None
if cfg.get("resume_from") and os.path.exists(cfg["resume_from"]):
    ck = torch.load(cfg["resume_from"], map_location=device, weights_only=False)
    raw_model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    step, tokens_seen = ck["step"], ck["tokens"]
    train_seconds_before, best_val = ck["train_seconds"], ck.get("best_val", float("inf"))
    warmup_end_progress = ck.get("warmup_end_progress")
    log("resumed", path=cfg["resume_from"], step=step, tokens=tokens_seen)


# ---------------------------------------------------------------------------
# schedule: warmup in steps, then decay over wall-clock progress to the deadline
# ---------------------------------------------------------------------------
if cfg.get("probe"):                     # runtime probe: fixed training window from here
    DEADLINE = min(DEADLINE, time.time() + cfg["probe"]["train_minutes"] * 60)
t_start = time.time()
schedule_seconds = (cfg["schedule_hours"] * 3600 if cfg.get("schedule_hours")
                    else train_seconds_before + (DEADLINE - t_start))
max_tokens = TR.get("max_tokens")


def progress_now():
    p = (train_seconds_before + time.time() - t_start) / schedule_seconds
    if max_tokens:
        p = max(p, tokens_seen / max_tokens)
    return p


def lr_at(step, p):
    global warmup_end_progress
    lr, min_lr, warm = TR["lr"], TR["min_lr"], TR["warmup_steps"]
    if step < warm:
        return lr * (step + 1) / warm
    if warmup_end_progress is None:
        warmup_end_progress = min(p, 0.99)
    q = min(1.0, max(0.0, (p - warmup_end_progress) / (1 - warmup_end_progress)))
    kind = TR.get("schedule", "cosine")
    if kind == "cosine":
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * q))
    if kind == "linear":
        return lr - (lr - min_lr) * q
    return lr


# ---------------------------------------------------------------------------
# eval + checkpoints
# ---------------------------------------------------------------------------
val_starts = np.random.default_rng(1234).integers(0, len(val_mm) - T - 1, TR["eval_batches"] * mb)


@torch.no_grad()
def evaluate():
    raw_model.eval()
    losses = []
    for i in range(0, len(val_starts), mb):
        x, y = windows(val_mm, val_starts[i:i + mb])
        losses.append(loss_fn(raw_model, x.to(device), y.to(device)).item())
    raw_model.train()
    return float(np.mean(losses))


def save(path, full):
    state = {"model": raw_model.state_dict(), "config": cfg, "step": step, "tokens": tokens_seen,
             "train_seconds": train_seconds_before + time.time() - t_start, "best_val": best_val,
             "warmup_end_progress": warmup_end_progress}
    if full:
        state.update(opt=opt.state_dict(), scaler=scaler.state_dict())
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


def do_eval(tag):
    global best_val
    t0 = time.time()
    vl = evaluate()
    t_eval = time.time() - t0
    improved = vl < best_val
    if improved:
        best_val = vl
        save(os.path.join(OUT, "best.pt"), full=False)
    save(os.path.join(OUT, "last.pt"), full=True)
    log(tag, step=step, tokens=tokens_seen, val_loss=round(vl, 4), val_ppl=round(math.exp(vl), 2),
        best=improved, hours=round((time.time() - t_start) / 3600, 2),
        eval_s=round(t_eval, 1), save_s=round(time.time() - t0 - t_eval, 1))


# ---------------------------------------------------------------------------
# train
# ---------------------------------------------------------------------------
pf = Prefetcher(mb, seed + step)
raw_model.train()
last_eval = time.time()
last_log, tok_at_last_log = time.time(), tokens_seen
eval_every = TR["eval_every_min"] * 60
bad_logs = 0
log("training", deadline_in_h=round((DEADLINE - t_start) / 3600, 2), schedule_hours=round(schedule_seconds / 3600, 2))


def train_step(lr):
    for g in opt.param_groups:
        g["lr"] = lr
    loss_sum = torch.zeros((), device=device)
    for _ in range(accum):
        x, y = pf.next()
        loss = loss_fn(model, x, y) / accum
        scaler.scale(loss).backward()
        loss_sum += loss.detach()
    if TR.get("grad_clip"):
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), TR["grad_clip"]).item()
    else:
        gnorm = float("nan")
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return loss_sum, gnorm


while True:
    p = progress_now()
    if time.time() >= DEADLINE or p >= 1.0:
        break
    lr = lr_at(step, p)
    try:
        loss_sum, gnorm = train_step(lr)
    except torch.cuda.OutOfMemoryError:
        # can only happen in the first steps (optimizer state / compiled graphs); shrink the micro-batch
        opt.zero_grad(set_to_none=True)
        gc.collect(); torch.cuda.empty_cache()
        smaller = [d for d in candidates if d < mb]
        if not smaller:
            log("OOM at micro-batch 1; stopping training", step=step, error=traceback.format_exc()[-1500:])
            break
        mb = smaller[0]
        accum = batch_seqs // mb
        pf = Prefetcher(mb, seed + step + 1)
        log("OOM: reduced micro-batch", micro_batch=mb, grad_accum=accum, step=step)
        continue
    except Exception:
        opt.zero_grad(set_to_none=True)
        if model is not raw_model:
            model = raw_model
            log("error under torch.compile; continuing in eager mode", step=step, error=traceback.format_exc()[-1500:])
            continue
        log("training error; stopping training and moving on to generation", step=step,
            error=traceback.format_exc()[-1500:])
        break
    step += 1
    tokens_seen += batch_seqs * T

    if step % TR.get("log_every", 50) == 0:
        loss_sum = loss_sum.item()
        now = time.time()
        log("train", step=step, loss=round(loss_sum, 4), lr=lr, grad_norm=round(gnorm, 3),
            tok_per_s=int((tokens_seen - tok_at_last_log) / (now - last_log)),
            tokens_M=round(tokens_seen / 1e6, 1), progress=round(p, 4),
            mem_GB=round(torch.cuda.max_memory_allocated() / 1e9, 2) if CUDA else 0.0)
        last_log, tok_at_last_log = now, tokens_seen
        bad_logs = 0 if math.isfinite(loss_sum) else bad_logs + 1
        if bad_logs >= 3:
            log("loss non-finite for 3 log intervals; stopping training (generation uses the best checkpoint)", step=step)
            break
    if time.time() - last_eval >= eval_every:
        do_eval("eval")
        last_eval = time.time()

try:
    do_eval("final_eval")
except Exception:
    log("final eval failed", error=traceback.format_exc()[-1500:])
    save(os.path.join(OUT, "last.pt"), full=False)
log("training done", steps=step, tokens=tokens_seen, best_val=round(best_val, 4))

# ---------------------------------------------------------------------------
# generation -> prediction.json (+ held-out val prompts for local scoring)
# ---------------------------------------------------------------------------
del pf, opt
if CUDA:
    torch.cuda.empty_cache()
which = GEN.get("checkpoint", "best")
if not os.path.exists(os.path.join(OUT, f"{which}.pt")):
    log(f"{which}.pt missing (no finite val loss?); falling back to last.pt")
    which = "last"
ck = torch.load(os.path.join(OUT, f"{which}.pt"), map_location=device, weights_only=False)
raw_model.load_state_dict(ck["model"])
raw_model.eval()
if not TR.get("fast_attention", True):
    fastattn.patch_model(raw_model)          # KV cache needs the patched forward


class ProbeTokenizer:
    """Stand-in for the real tokenizer in probe mode: every prompt encodes to `prompt_len` random ids
    and no token ever ends a word, so each batch runs the full max_new_tokens (worst case)."""
    n_special, eow_id = 32, 0

    def __init__(self, vocab_size, prompt_len):
        self.token_eow = [False] * vocab_size
        self.token_bytes = [b"x"] * vocab_size
        self._ids = np.random.default_rng(0).integers(0, vocab_size, prompt_len).tolist()

    def encode(self, text):
        return self._ids

    def decode(self, ids):
        return ""


if cfg.get("probe"):
    tokenizer = ProbeTokenizer(V, T - GEN["max_new_tokens"])
else:
    tokenizer = ByteBPE.from_json(cfg["tokenizer_path"])


def run_gen(prompts):
    bs = GEN["batch_size"]
    while True:
        try:
            return fastattn.generate_words(
                raw_model, tokenizer, prompts, max_seq_len=T, target_words=GEN["target_words"],
                max_new_tokens=GEN["max_new_tokens"], batch_size=bs, temperature=GEN["temperature"],
                top_k=GEN.get("top_k"), top_p=GEN.get("top_p"), repetition_penalty=GEN.get("repetition_penalty", 1.0),
                device=device, amp_dtype=AMP if CUDA else torch.float32)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs //= 2
            assert bs >= 1
            log("generation OOM, retrying", batch_size=bs)


if cfg.get("probe"):
    times = []
    for _ in range(2):                       # the first batch includes CUDA warm-up
        if CUDA:
            torch.cuda.synchronize()
        t0 = time.time()
        run_gen(["probe"] * GEN["batch_size"])
        if CUDA:
            torch.cuda.synchronize()
        times.append(time.time() - t0)
    log("probe_generation", batch_size=GEN["batch_size"], prompt_tokens=T - GEN["max_new_tokens"],
        new_tokens=GEN["max_new_tokens"], first_batch_s=round(times[0], 2), batch_s=round(times[1], 2))
    log("all done")
    sys.exit(0)

torch.manual_seed(GEN.get("seed", 0))
test = json.load(open(cfg["test_path"], encoding="utf-8"))
ids = list(test.keys())
t0 = time.time()
preds = run_gen([test[k]["text"] for k in ids])
prediction = {k: {"text": p} for k, p in zip(ids, preds)}
with open(os.path.join(OUT, "prediction.json"), "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)
short = sum(len(p.split()) < GEN["target_words"] for p in preds)
log("prediction.json written", n=len(preds), minutes=round((time.time() - t0) / 60, 1), under_target_words=short,
    checkpoint=which)

vp_path = os.path.join(cfg["data_dir"], "val_prompts.json")
n_val = GEN.get("n_val_prompts", 0)
if n_val and os.path.exists(vp_path) and SESSION_END - time.time() > 2 * (time.time() - t0) + 600:
    try:
        vp = json.load(open(vp_path, encoding="utf-8"))
        keys = list(vp)[:n_val]
        vpreds = run_gen([vp[k]["text"] for k in keys])
        with open(os.path.join(OUT, "val_generations.json"), "w", encoding="utf-8") as f:
            json.dump({k: {"prediction": p, "reference": vp[k]["reference"]} for k, p in zip(keys, vpreds)},
                      f, ensure_ascii=False, indent=2)
        log("val_generations.json written", n=len(keys))
    except Exception:
        log("val generation failed (prediction.json is unaffected)", error=traceback.format_exc()[-1500:])
log("all done")

## Stage 2 — sanity checks (CPU)

In [ ]:
# sanity checks before spending GPU hours (CPU, ~1 min)
#   * parameter counts (limit 50M-300M)
#   * fastattn (SDPA + KV cache) matches your original forward, for the exact configs above
import importlib
import fastattn, arch1, arch2, arch3
for m in (fastattn, arch1, arch2, arch3):
    importlib.reload(m)

for tag, spec in (("A", MODEL_A), ("B", MODEL_B)):
    mod = importlib.import_module(spec["module"])
    make = lambda: mod.DecoderLM(mod.LMConfig(**spec["model"]))
    model = make()
    n = sum(p.numel() for p in model.parameters())            # tied embedding counted once
    emb = model.tok_emb.weight.numel()
    print(f"model {tag} ({spec['module']}): {n / 1e6:.2f}M params  (embedding {emb / 1e6:.2f}M, non-embedding {(n - emb) / 1e6:.2f}M)")
    assert 50e6 <= n <= 300e6, "Phase 1 requires 50M-300M parameters"
    T = spec["model"]["max_seq_len"]
    assert GEN["max_new_tokens"] < T, f"GEN max_new_tokens ({GEN['max_new_tokens']}) must be < max_seq_len ({T})"
    print(f"   generation keeps the last {T - GEN['max_new_tokens']} prompt tokens "
          f"(a 512-word prompt is ~{int(512 * 1.2)} tokens at ~1.2 tok/word)")
    del model
    err_full, err_cache = fastattn.check_equivalence(make, spec["model"]["vocab_size"], seq_len=32)
    print(f"   fused attention vs your forward: rel. max|diff| {err_full:.1e}; KV-cached decoding: {err_cache:.1e}  OK")

## Stage 2b — runtime probe on 2×T4 (~20–25 min; run this before committing to a 12 h run)
Runs the **real** `train_lm.py` with your exact `MODEL_A/B` + `TRAIN_A/B` on both GPUs in parallel, but on synthetic
random tokens (throughput does not depend on token content), so it needs neither the corpus nor the tokenizer.
Measures: micro-batch/grad-accum picked, peak memory, `torch.compile` time, steady tokens/s per GPU and whether it
drops as the T4s heat up (plus nvidia-smi clocks/temperature/power), eval + checkpoint time, checkpoint size,
worst-case generation time. Then projects the 12 h run: time to `max_tokens`, tokens reachable before the deadline,
and whether generation fits in `GEN_RESERVE_MIN`.
**Not measured:** the data build (stage 3 prints its real timing) and prompt encoding on CPU.
With `PROBE_ONLY = True` the notebook stops after the report (the cell raises `ProbeOnlyStop` on purpose);
set `PROBE_ONLY = False` (or `RUN_PROBE = False`) for the real run.

In [ ]:
# runtime probe: real train_lm.py, synthetic tokens, both GPUs in parallel, then a projection for the full run
RUN_PROBE = False
PROBE_ONLY = False              # True: stop after the report. False: continue into the full pipeline (probe time is then spent from this session)
PROBE_TRAIN_MIN = 12            # timed training window per GPU, after micro-batch search + torch.compile
PROBE_EVAL_EVERY_MIN = 4        # eval + checkpoint save a few times, to time them
DATA_BUILD_H_EST = 0.0 if PREBUILT_DATA_DIR else 1.0   # a guess, NOT measured here (stage 3 prints the real figure)
KAGGLE_WORKING_GB = 19.5        # /kaggle/working output limit


class ProbeOnlyStop(Exception):
    pass


PROBE_DIR = os.path.join(WORK, "probe")


def _read_log(tag):
    p = os.path.join(PROBE_DIR, f"run_{tag}", "log.jsonl")
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []


def _median(xs):
    return float(np.median(xs)) if len(xs) else float("nan")


if RUN_PROBE and not DATA_ONLY:
    from huggingface_hub import hf_hub_download
    t_probe = time.time()
    shutil.rmtree(PROBE_DIR, ignore_errors=True)
    probe_data = os.path.join(PROBE_DIR, "data")
    os.makedirs(probe_data)
    _rng = np.random.default_rng(SEED)
    _rng.integers(0, VOCAB_SIZE, 20_000_000, dtype=np.uint16).tofile(os.path.join(probe_data, "train.bin"))
    _rng.integers(0, VOCAB_SIZE, 2_000_000, dtype=np.uint16).tofile(os.path.join(probe_data, "val.bin"))

    def _probe_cfg(tag, spec, train):
        cfg = {
            "tag": tag, "module": spec["module"], "model": spec["model"], "init_std": spec["init_std"],
            "data_dir": probe_data, "out_dir": os.path.join(PROBE_DIR, f"run_{tag}"),
            "tokenizer_path": None, "test_path": None,
            "deadline_unix": SESSION_START + SESSION_HOURS * 3600 - SAFETY_MIN * 60,
            "session_end_unix": SESSION_START + SESSION_HOURS * 3600 - SAFETY_MIN * 60, "schedule_hours": None,
            "resume_from": None, "probe": {"train_minutes": PROBE_TRAIN_MIN},
            "train": dict(train, eval_every_min=PROBE_EVAL_EVERY_MIN, log_every=20, max_tokens=None),   # full window
            "gen": GEN,
        }
        path = os.path.join(WORK, f"probe_{tag}.json")
        json.dump(cfg, open(path, "w"), indent=2)
        return path

    def _launch(tag, gpu):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS="1", PYTHONUNBUFFERED="1")
        out = open(os.path.join(PROBE_DIR, f"probe_{tag}.log"), "w")
        return subprocess.Popen([sys.executable, "train_lm.py", _probe_cfg(tag, *SPECS[tag])],
                                cwd=WORK, env=env, stdout=out, stderr=subprocess.STDOUT)

    gpu_samples = []
    _Q = "index,temperature.gpu,clocks.sm,clocks.max.sm,power.draw,power.limit,utilization.gpu,memory.used"

    def _poll_gpus():
        try:
            out = subprocess.run(["nvidia-smi", f"--query-gpu={_Q}", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=20).stdout
        except Exception:
            return
        def num(x):                                   # "[N/A]" on some fields/drivers
            try:
                return float(x)
            except ValueError:
                return float("nan")

        for line in out.strip().splitlines():
            f = [num(x.strip()) for x in line.split(",")]
            if len(f) == 8:
                gpu_samples.append(dict(t=time.time() - t_probe, gpu=int(f[0]), temp=f[1], sm=f[2], sm_max=f[3],
                                        power=f[4], power_limit=f[5], util=f[6], mem_mb=f[7], load=os.getloadavg()[0]))

    def _wait(procs):
        last_print = time.time()
        while any(p.poll() is None for p in procs.values()):
            _poll_gpus()
            time.sleep(15)
            if time.time() - last_print > 120:
                last_print = time.time()
                print(f"{(time.time() - t_probe) / 60:5.1f} min", flush=True)
                for tag in procs:
                    recs = _read_log(tag)
                    if recs:
                        print(f"  [{tag}] {json.dumps(recs[-1])[:220]}", flush=True)
        for tag, p in procs.items():
            if p.returncode:
                print(f"probe {tag} exited with code {p.returncode}; last lines:")
                print("\n".join(open(os.path.join(PROBE_DIR, f"probe_{tag}.log")).read().splitlines()[-25:]))

    SPECS = {"A": (MODEL_A, TRAIN_A), "B": (MODEL_B, TRAIN_B)}
    print(f"Probe: {PROBE_TRAIN_MIN} min timed training per model + compile/eval/generation, "
          f"{'both GPUs in parallel' if N_GPUS >= 2 else 'ONE GPU: sequential'}", flush=True)
    if N_GPUS >= 2:
        _wait({"A": _launch("A", 0), "B": _launch("B", 1)})
    else:
        _wait({"A": _launch("A", 0)})
        _wait({"B": _launch("B", 0)})
    probe_minutes = (time.time() - t_probe) / 60

    # ------------------------------------------------------------------ report
    test_n = len(json.load(open(hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                                repo_type="dataset", token=HF_TOKEN), encoding="utf-8")))
    report = {"probe_minutes": round(probe_minutes, 1), "n_gpus": N_GPUS, "cpus": os.cpu_count(), "models": {}}
    print("\n" + "=" * 100 + f"\nPROBE REPORT  (probe took {probe_minutes:.1f} min, {os.cpu_count()} CPUs, test prompts: {test_n})\n" + "=" * 100)
    for tag in ("A", "B"):
        recs = _read_log(tag)
        by = lambda m: [r for r in recs if r["msg"] == m]
        problems = [r for r in recs if any(w in r["msg"] for w in ("OOM", "error", "failed", "non-finite", "stopping"))]
        tr = by("train")
        if not tr:
            print(f"\nmodel {tag}: NO training steps logged -> see {PROBE_DIR}/probe_{tag}.log")
            for r in problems:
                print("   ", json.dumps(r)[:400])
            report["models"][tag] = {"failed": True}
            continue
        m, b = by("model")[0], by("batch")[-1]
        comp = by("torch.compile ok")
        setup_s = by("training")[0]["t"] - m["t"]
        tps = [r["tok_per_s"] for r in tr[1:]] or [r["tok_per_s"] for r in tr]   # first interval includes warm-up
        third = max(1, len(tps) // 3)
        tps_first, tps_last = _median(tps[:third]), _median(tps[-third:])
        tps_use = min(_median(tps), tps_last)                                      # conservative: throttled speed
        evals = by("eval") + by("final_eval")
        eval_s = _median([r["eval_s"] for r in evals])
        save_s = _median([r["save_s"] for r in evals])
        gen = by("probe_generation")
        run_dir = os.path.join(PROBE_DIR, f"run_{tag}")
        ck_gb = {f: os.path.getsize(os.path.join(run_dir, f)) / 1e9 for f in ("last.pt", "best.pt")
                 if os.path.exists(os.path.join(run_dir, f))}
        train = SPECS[tag][1]

        # projection for a full 12 h session
        pre_h = (t_probe - SESSION_START) / 3600 + DATA_BUILD_H_EST + setup_s / 3600
        if not PROBE_ONLY:
            pre_h += probe_minutes / 60
        budget_h = (SESSION_HOURS * 60 - SAFETY_MIN - GEN_RESERVE_MIN) / 60 - pre_h
        eval_frac = (eval_s + save_s) / (train["eval_every_min"] * 60)
        eff_tps = tps_use * (1 - eval_frac)
        tokens_budget = eff_tps * budget_h * 3600
        cap = train.get("max_tokens")
        step_tokens = b["tokens_per_step"]
        gen_batch_s = gen[0]["batch_s"] if gen else float("nan")
        n_test_batches = math.ceil(test_n / GEN["batch_size"])
        n_val_batches = math.ceil(min(GEN.get("n_val_prompts", 0), N_VAL_PROMPTS) / GEN["batch_size"])
        gen_test_min = gen_batch_s * n_test_batches / 60
        gen_val_min = gen_batch_s * n_val_batches / 60
        end_min = (eval_s + save_s) / 60 + gen_test_min            # final eval + test generation

        print(f"\nmodel {tag} ({m['module']}, {m['params_M']}M params, seq {m['seq_len']})")
        print(f"  batch        : {b['global_seqs']} seqs x {m['seq_len']} = {step_tokens:,} tokens/step | micro-batch {b['micro_batch']} "
              f"x accum {b['grad_accum']} | peak memory {max(r.get('mem_GB', 0) for r in tr):.1f} GB (of ~15 GB)")
        comp_txt = (f"compile {comp[0]['seconds']:.0f} s" if comp else
                    "compile FAILED -> eager (slower)" if train.get("compile") else "compile off")
        print(f"  setup        : {setup_s / 60:.1f} min (micro-batch search + {comp_txt})")
        print(f"  throughput   : median {_median(tps):,.0f} tok/s | first third {tps_first:,.0f} -> last third {tps_last:,.0f} "
              f"({(tps_last / tps_first - 1) * 100:+.1f}%) | {tps_use / step_tokens:.2f} steps/s")
        print(f"  eval + save  : {eval_s:.1f} s + {save_s:.1f} s every {train['eval_every_min']} min in the real run "
              f"({eval_frac:.1%} overhead) | checkpoints " + ", ".join(f"{k} {v:.2f} GB" for k, v in ck_gb.items()))
        print(f"  generation   : {gen_batch_s:.1f} s per batch of {GEN['batch_size']} (worst case: {GEN['max_new_tokens']} new tokens) "
              f"-> test {gen_test_min:.1f} min + val prompts {gen_val_min:.1f} min")
        print(f"  12 h run     : training budget ~{budget_h:.2f} h (after ~{pre_h:.2f} h setup incl. {DATA_BUILD_H_EST:.1f} h data-build guess)")
        if cap:
            t_cap_h = cap / eff_tps / 3600
            if t_cap_h <= budget_h:
                print(f"                 max_tokens {cap / 1e6:.0f}M reached after ~{t_cap_h:.2f} h "
                      f"({cap / step_tokens:,.0f} steps) -> ~{budget_h - t_cap_h:.2f} h of the training budget unused")
            else:
                print(f"                 deadline comes first: ~{tokens_budget / 1e6:.0f}M of max_tokens {cap / 1e6:.0f}M "
                      f"({tokens_budget / cap:.0%}); the LR schedule compresses to the deadline")
        else:
            print(f"                 no max_tokens: schedule runs to the deadline -> ~{tokens_budget / 1e6:.0f}M tokens "
                  f"({tokens_budget / step_tokens:,.0f} steps)")
        ok_gen = end_min < GEN_RESERVE_MIN
        print(f"                 final eval + test generation ~{end_min:.1f} min vs GEN_RESERVE_MIN {GEN_RESERVE_MIN} "
              f"-> {'OK' if ok_gen else 'DOES NOT FIT: raise GEN_RESERVE_MIN or lower max_new_tokens/batch cost'}"
              f"{'' if end_min < 0.7 * GEN_RESERVE_MIN else ' (tight)'}")
        for r in problems:
            print("  WARNING:", json.dumps(r)[:300])
        report["models"][tag] = dict(
            params_M=m["params_M"], micro_batch=b["micro_batch"], grad_accum=b["grad_accum"], tokens_per_step=step_tokens,
            peak_mem_GB=max(r.get("mem_GB", 0) for r in tr), setup_min=round(setup_s / 60, 1),
            compile_s=comp[0]["seconds"] if comp else None, tok_per_s_median=round(_median(tps)),
            tok_per_s_first_third=round(tps_first), tok_per_s_last_third=round(tps_last),
            eval_s=eval_s, save_s=save_s, checkpoint_GB=ck_gb, gen_batch_s=gen_batch_s,
            gen_test_min_worst=round(gen_test_min, 1), train_budget_h=round(budget_h, 2),
            tokens_in_budget_M=round(tokens_budget / 1e6), gen_fits=ok_gen)

    print("\nGPUs (nvidia-smi every 15 s while the probe ran; first vs last third of busy samples):")
    if not gpu_samples:
        print("  (nvidia-smi gave no samples)")
    for g in sorted({s["gpu"] for s in gpu_samples}):
        busy = [s for s in gpu_samples if s["gpu"] == g and s["util"] >= 50]
        if not busy:
            print(f"  GPU{g}: never above 50% utilisation while sampled")
            continue
        k = max(1, len(busy) // 3)
        f_, l_ = busy[:k], busy[-k:]
        print(f"  GPU{g}: SM clock {_median([s['sm'] for s in f_]):.0f} -> {_median([s['sm'] for s in l_]):.0f} MHz "
              f"(max {busy[0]['sm_max']:.0f}) | temp {_median([s['temp'] for s in f_]):.0f} -> {_median([s['temp'] for s in l_]):.0f} C "
              f"(peak {max(s['temp'] for s in busy):.0f}) | power {_median([s['power'] for s in busy]):.0f}/{busy[0]['power_limit']:.0f} W "
              f"| util {_median([s['util'] for s in busy]):.0f}% | mem {max(s['mem_mb'] for s in busy) / 1024:.1f} GB")
    loads = [s["load"] for s in gpu_samples]
    if loads:
        print(f"  CPU load average (1 min): median {_median(loads):.1f} on {os.cpu_count()} CPUs "
              f"({'CPU-bound data loading possible' if _median(loads) > 0.9 * (os.cpu_count() or 4) else 'OK'})")
    report["gpus"] = gpu_samples

    ck_total = sum(sum(v.get("checkpoint_GB", {}).values()) for v in report["models"].values())
    data_gb = 2 * TOTAL_TRAIN_TOKENS / 1e9 + 0.1
    print(f"\nDisk: full run writes ~{data_gb + ck_total:.1f} GB to /kaggle/working (train.bin ~{data_gb:.1f} GB + checkpoints "
          f"{ck_total:.1f} GB) of ~{KAGGLE_WORKING_GB} GB; free now {shutil.disk_usage(WORK).free / 1e9:.0f} GB")
    json.dump(report, open(os.path.join(WORK, "probe_report.json"), "w"), indent=2)
    for f in glob.glob(os.path.join(PROBE_DIR, "run_*", "*.pt")) + glob.glob(os.path.join(probe_data, "*.bin")):
        os.remove(f)                                  # keep logs, free the disk for the real run
    print(f"Report saved to {WORK}/probe_report.json; probe logs in {PROBE_DIR}")

    if PROBE_ONLY:
        raise ProbeOnlyStop("Probe finished (PROBE_ONLY = True): read the report above. "
                            "Set PROBE_ONLY = False (or RUN_PROBE = False) for the full run.")


## Stage 3 — corpus download, legal BPE, pretraining data (CPU; skipped when prebuilt data is attached)
Run this once in a **CPU-only session** with `DATA_ONLY = True` (no GPU quota used), commit, then attach the output
to the GPU session (`PREBUILT_DATA_DIR = "auto"` finds it). Timings are printed per step and saved to `manifest.json`.

In [ ]:
# shared helpers for stage 3 (download, streaming readers, random-seek sampler)
import importlib
import multiprocessing as mp
from collections import Counter
from huggingface_hub import hf_hub_download
import a1_tokenizer
importlib.reload(a1_tokenizer)
from a1_tokenizer import ByteBPE

STAGE3_START = time.time()
N_WORKERS = os.cpu_count() or 4
NEED_TOKENIZER = not os.path.exists(TOKENIZER_PATH)
NEED_DATA = not os.path.exists(os.path.join(DATA_DIR, "train.bin"))
assert not (PREBUILT_DATA_DIR and (NEED_TOKENIZER or NEED_DATA)), f"{PREBUILT_DATA_DIR} lacks legal_bpe.json / train.bin"


def fetch(fname):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)


test_path = fetch(TEST_FILE)
test_data = json.load(open(test_path, encoding="utf-8"))
test_texts = [r["text"] for r in test_data.values()]
print(f"Test prompts: {len(test_texts):,}")

SOURCES = {}
if NEED_TOKENIZER or NEED_DATA:
    t0 = time.time()
    SOURCES = {name: fetch(f) for name, f in TRAIN_FILES.items()}
    print(f"Downloaded corpus in {(time.time() - t0) / 60:.1f} min")


def normalize_ws(text):
    return " ".join(text.split())


def doc_hash(text):
    return int(hashlib.md5(text[:2000].encode("utf-8", errors="ignore")).hexdigest(), 16)


def _parse_line(line):
    line = line.strip().rstrip(",")
    if not line.startswith("{"):
        return None
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        return None


def iter_texts(path):
    """Stream the "text" field of a one-record-per-line JSON array file."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = _parse_line(line)
            if rec and rec.get("text"):
                yield rec["text"]


def sample_texts_random_seek(path, n_chars, seed):
    """Random byte seeks -> resync to the next line -> one record. Spreads the sample over the whole file."""
    rng = random.Random(seed)
    size = os.path.getsize(path)
    texts, seen, total, misses = [], set(), 0, 0
    with open(path, "rb") as f:
        while total < n_chars and misses < 5000:
            f.seek(rng.randrange(0, size))
            f.readline()
            raw = f.readline()
            rec = _parse_line(raw.decode("utf-8", errors="ignore")) if raw else None
            text = rec.get("text") if rec else None
            if not text:
                misses += 1
                continue
            h = doc_hash(text)
            if h in seen:
                misses += 1
                continue
            seen.add(h)
            texts.append(text)
            total += len(text)
            misses = 0
    if total < n_chars:
        print(f"  WARNING: {os.path.basename(path)} gave only {total / 1e6:.1f}M of {n_chars / 1e6:.1f}M chars")
    return texts


def fertility(tok, texts):
    n_tok = sum(len(tok.encode(t)) for t in texts)
    n_words = sum(len(t.split()) for t in texts)
    n_chars = sum(len(normalize_ws(t)) for t in texts)
    return n_tok / n_words, n_chars / n_tok

In [ ]:
# train the 32k legal BPE (skipped if legal_bpe.json exists)
os.makedirs(DATA_DIR, exist_ok=True)
if NEED_TOKENIZER:
    t0 = time.time()
    counts = Counter()
    for name, share in TOK_SOURCE_MIX.items():
        texts = sample_texts_random_seek(SOURCES[name], int(share * TOK_TRAIN_CHARS), seed=SEED)
        counts.update(ByteBPE.count_words(texts))
        print(f"  [{name}] {len(texts):,} docs sampled | {len(counts):,} distinct words so far")
        del texts
    print(f"Word counting: {(time.time() - t0) / 60:.1f} min")
    t0 = time.time()
    tokenizer = ByteBPE(vocab_size=VOCAB_SIZE).fit(
        word_counts=counts, target_vocab_size=VOCAB_SIZE, min_count=a1_tokenizer.MIN_COUNT,
        min_word_freq=TOK_MIN_WORD_FREQ)
    print(f"BPE training: {(time.time() - t0) / 60:.1f} min")
    tokenizer.to_json(TOKENIZER_PATH)
    del counts
else:
    tokenizer = ByteBPE.from_json(TOKENIZER_PATH)
    print("Loaded", TOKENIZER_PATH)

V = tokenizer.get_vocab_size()
EOT_ID = tokenizer.eot_id
assert V < 2 ** 16

fert, cpt = fertility(tokenizer, test_texts)
roundtrip = np.mean([tokenizer.decode(tokenizer.encode(t)) == normalize_ws(t) for t in test_texts[:300]])
print(f"vocab {V} | test fertility {fert:.4f} tok/word | {cpt:.2f} chars/token | round-trip {roundtrip:.1%}")
print("(earlier HF byte-level BPE 32k: 1.2006 tok/word; GPT-2 50k: 1.2398)")
print([tokenizer.itos[i] for i in tokenizer.encode(test_texts[0][:300])])

In [ ]:
# build train.bin / val.bin / val_prompts.json with the legal BPE (skipped if they exist)
# Same recipe as your earlier pipeline: whitespace-normalised docs; all Supreme Court (repeated up to
# SC_MAX_EPOCHS, hash-split slice held out for validation); Acts up to ACTS_MAX_EPOCHS; the rest filled with
# the most SC-like High Court / Tribunal docs (DSIR-style unigram score). Encoding runs on all CPU cores.
# Change vs. before: HC/Tribunals are scored on a random sample of HT_SCAN_FACTOR x the needed tokens instead
# of the whole 17 GB, which keeps this stage around an hour in pure Python.
SCRATCH = "/tmp/cs781_scratch"


_encode_doc = a1_tokenizer.encode_array_worker   # runs in worker processes: (uint16 ids, word count)


def encode_docs(name, texts, score_profile=None):
    """Encode docs to a scratch .bin; returns offsets, per-doc DSIR scores and stats."""
    bin_path = os.path.join(SCRATCH, f"{name}.bin")
    lengths, scores, stats = [], [], Counter()
    t0 = time.time()
    with open(bin_path, "wb") as out, tokenizer.pool(N_WORKERS) as pool:
        for arr, n_words in pool.imap(_encode_doc, texts, chunksize=8):
            stats["docs_seen"] += 1
            if name != "supreme_court" and len(arr) / n_words > MAX_FERTILITY:
                stats["dropped_fertility"] += 1
                continue
            arr.tofile(out)
            lengths.append(len(arr))
            if score_profile is not None:
                i64 = arr.astype(np.int64)
                scores.append(float(profiles["supreme_court"][i64].mean() - score_profile[i64].mean()))
            if len(lengths) % 50_000 == 0:
                print(f"  [{name}] {len(lengths):,} docs | {sum(lengths) / 1e6:,.0f}M tokens | "
                      f"{sum(lengths) / (time.time() - t0) / 1e6:.2f}M tok/s", flush=True)
    offsets = np.concatenate([[0], np.cumsum(np.asarray(lengths, dtype=np.int64))])
    stats.update(docs_kept=len(lengths), unique_tokens=int(offsets[-1]), encode_minutes=round((time.time() - t0) / 60, 2))
    print(f"[{name}] {dict(stats)}", flush=True)
    return {"bin": bin_path, "offsets": offsets, "scores": np.asarray(scores, np.float32), "stats": dict(stats)}


def clean_stream(texts, stats, val_sink=None):
    for raw in texts:
        text = normalize_ws(raw)
        if text.count(" ") + 1 < MIN_WORDS:
            stats["dropped_short"] += 1
            continue
        if val_sink is not None and doc_hash(text) % 100 < SC_VAL_PERCENT:
            val_sink.append(text)
            continue
        stats["chars"] += len(text)
        yield text


if NEED_DATA:
    os.makedirs(SCRATCH, exist_ok=True)
    free_gb = shutil.disk_usage(SCRATCH).free / 1e9
    print(f"Free scratch disk: {free_gb:.1f} GB")
    t_all = time.time()

    # unigram profiles for the DSIR-style score
    def unigram_logprobs(texts):
        counts = np.zeros(V, dtype=np.float64)
        with tokenizer.pool(N_WORKERS) as pool:
            for arr, _ in pool.imap(_encode_doc, (normalize_ws(t) for t in texts), chunksize=8):
                counts += np.bincount(arr, minlength=V)
        return np.log((counts + 1) / (counts.sum() + V)).astype(np.float32)

    profiles = {s: unigram_logprobs(sample_texts_random_seek(p, PROFILE_CHARS, seed=SEED + 2)) for s, p in SOURCES.items()}
    print(f"profiles built ({(time.time() - t_all) / 60:.1f} min)")

    encoded, stream_stats, val_texts = {}, {}, []
    for name in ("supreme_court", "acts"):
        stream_stats[name] = Counter()
        sink = val_texts if name == "supreme_court" else None
        encoded[name] = encode_docs(name, clean_stream(iter_texts(SOURCES[name]), stream_stats[name], sink))
    sc_chars_per_token = stream_stats["supreme_court"]["chars"] / encoded["supreme_court"]["stats"]["unique_tokens"]

    rng = np.random.default_rng(SEED)

    def doc_len(src, i):
        off = encoded[src]["offsets"]
        return int(off[i + 1] - off[i])

    def repeat_docs(src, budget):
        """Whole epochs over all docs, then a random subset for the fractional epoch."""
        n_docs = len(encoded[src]["offsets"]) - 1
        unique = encoded[src]["stats"]["unique_tokens"]
        full = int(budget // unique) if unique else 0
        picks, used = [(src, i) for _ in range(full) for i in range(n_docs)], full * unique
        for i in rng.permutation(n_docs):
            if used >= budget:
                break
            picks.append((src, int(i)))
            used += doc_len(src, int(i))
        return picks, used

    sc_budget = min(SC_MAX_EPOCHS * encoded["supreme_court"]["stats"]["unique_tokens"], SC_MAX_SHARE * TOTAL_TRAIN_TOKENS)
    acts_budget = min(ACTS_MAX_EPOCHS * encoded["acts"]["stats"]["unique_tokens"], ACTS_MAX_SHARE * TOTAL_TRAIN_TOKENS)
    sc_picks, sc_used = repeat_docs("supreme_court", sc_budget)
    acts_picks, acts_used = repeat_docs("acts", acts_budget)
    rest_budget = max(0, TOTAL_TRAIN_TOKENS - sc_used - acts_used)

    # High Courts + Tribunals: random sample, scored, pooled ranking, each doc used at most once
    sizes = {s: os.path.getsize(SOURCES[s]) for s in ("high_courts", "tribunals")}
    scan_chars = HT_SCAN_FACTOR * rest_budget * sc_chars_per_token
    for s in sizes:
        stream_stats[s] = Counter()
        sample = sample_texts_random_seek(SOURCES[s], int(scan_chars * sizes[s] / sum(sizes.values())), seed=SEED + 3)
        encoded[s] = encode_docs(s, clean_stream(sample, stream_stats[s]), score_profile=profiles[s])
        del sample
    pool_rank = sorted(((float(sc), s, i) for s in sizes for i, sc in enumerate(encoded[s]["scores"])), key=lambda x: -x[0])
    ht_picks, ht_used, score_cutoff = [], 0, None
    for sc, s, i in pool_rank:
        if ht_used >= rest_budget:
            break
        ht_picks.append((s, i))
        ht_used += doc_len(s, i)
        score_cutoff = sc

    all_picks = sc_picks + acts_picks + ht_picks
    order = rng.permutation(len(all_picks))

    # train.bin: shuffled docs separated by </s>
    src_maps = {s: np.memmap(encoded[s]["bin"], dtype=np.uint16, mode="r") for s in encoded}
    train_len = sum(doc_len(s, i) + 1 for s, i in all_picks)
    train_mm = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="w+", shape=(train_len,))
    pos = 0
    for j in order:
        s, i = all_picks[j]
        off = encoded[s]["offsets"]
        n = int(off[i + 1] - off[i])
        train_mm[pos:pos + n] = src_maps[s][off[i]:off[i + 1]]
        train_mm[pos + n] = EOT_ID
        pos += n + 1
    train_mm.flush()
    del train_mm
    print(f"train.bin: {train_len:,} tokens")

    # val.bin + test-like val prompts (512-word prompt, 128-word reference) from held-out SC docs
    with tokenizer.pool(N_WORKERS) as pool:
        val_arrs = [a for a, _ in pool.imap(_encode_doc, val_texts, chunksize=8)]
    val_arr = np.concatenate([np.append(a, np.uint16(EOT_ID)) for a in val_arrs]).astype(np.uint16)
    val_arr.tofile(os.path.join(DATA_DIR, "val.bin"))
    val_prompts, prng = {}, np.random.default_rng(SEED + 7)
    for idx in prng.permutation(len(val_texts)):
        words = val_texts[idx].split()
        if len(words) < 640:
            continue
        st = int(prng.integers(0, len(words) - 640 + 1))
        val_prompts[str(len(val_prompts))] = {"text": " ".join(words[st:st + 512]), "reference": " ".join(words[st + 512:st + 640])}
        if len(val_prompts) >= N_VAL_PROMPTS:
            break
    json.dump(val_prompts, open(os.path.join(DATA_DIR, "val_prompts.json"), "w", encoding="utf-8"), ensure_ascii=False)

    used = {"supreme_court": sc_used, "acts": acts_used}
    for s, i in ht_picks:
        used[s] = used.get(s, 0) + doc_len(s, i)
    manifest = {
        "tokenizer": "legal_bpe.json", "vocab_size": V, "eot_id": EOT_ID, "dtype": "uint16",
        "train_tokens": int(train_len), "val_tokens": int(len(val_arr)), "val_prompts": len(val_prompts),
        "test_fertility": fert, "whitespace_normalised": True, "hc_trib_score_cutoff": score_cutoff,
        "sources": {s: {**encoded[s]["stats"], "tokens_used": int(used.get(s, 0)),
                        "share": round(used.get(s, 0) / max(sum(used.values()), 1), 4)} for s in encoded},
        "build_minutes": round((time.time() - t_all) / 60, 1),
    }
    json.dump(manifest, open(os.path.join(DATA_DIR, "manifest.json"), "w"), indent=2)
    del src_maps
    shutil.rmtree(SCRATCH, ignore_errors=True)
    print(json.dumps(manifest, indent=2))
else:
    print("Using existing data:", sorted(os.listdir(DATA_DIR)))
    print(open(os.path.join(DATA_DIR, "manifest.json")).read()[:1500])

In [ ]:
# DATA_ONLY: stop here after the data build (CPU-only session)
stage3_min = (time.time() - STAGE3_START) / 60
sizes = {f: os.path.getsize(os.path.join(DATA_DIR, f)) / 1e9 for f in sorted(os.listdir(DATA_DIR))}
print(f"Stage 3 wall time this session: {stage3_min:.1f} min (download + tokenizer + data build)")
print(f"Data dir: {DATA_DIR}")
for f, gb in sizes.items():
    print(f"  {f:24s} {gb:7.3f} GB")
timing_path = os.path.join(DATA_DIR, "build_timing.json")
if not PREBUILT_DATA_DIR:
    json.dump({"stage3_minutes": round(stage3_min, 1), "cpus": os.cpu_count(), "gpus": N_GPUS}, open(timing_path, "w"), indent=2)

if DATA_ONLY:
    shutil.rmtree(os.path.join(WORK, "probe"), ignore_errors=True)   # keep the committed output to the data only
    print("\n" + "=" * 100)
    print("DATA_ONLY = True: data build finished. Stage 4 (training) and stage 5 (submissions) will skip themselves,")
    print("so this version ends successfully and keeps its output. Next, in the GPU session:")
    print("  Add Input -> this notebook's output (or a Dataset made from it); Accelerator GPU T4 x2;")
    print("  DATA_ONLY = False (cell 4), RUN_PROBE = False (probe cell). PREBUILT_DATA_DIR = 'auto' finds the data.")
    print("=" * 100)


## Stage 4 — train A (GPU 0) ‖ B (GPU 1), then generate
Logs: `cs781_phase1/train_A.log`, `train_B.log`, and `run_*/log.jsonl`. Checkpoints: `run_*/best.pt` (weights) and `run_*/last.pt` (weights + optimiser, for `RESUME`).

In [ ]:
# train model A on GPU 0 and model B on GPU 1 in parallel; each then writes its prediction.json
if DATA_ONLY:
    print("DATA_ONLY = True -> stage 4 (training) skipped")
else:
    SESSION_END = SESSION_START + SESSION_HOURS * 3600 - SAFETY_MIN * 60
    train_deadline = SESSION_END - GEN_RESERVE_MIN * 60
    hours_left = (train_deadline - time.time()) / 3600
    print(f"Training budget: {hours_left:.2f} h per model (session ends in {(SESSION_END - time.time()) / 3600:.2f} h)")
    assert hours_left > 0.5, "not enough session time left for training"
    assert N_GPUS >= 1, "no GPU visible: set Accelerator to 'GPU T4 x2' in the notebook settings"


    def run_config(tag, spec, train, deadline):
        cfg = {
            "tag": tag, "module": spec["module"], "model": spec["model"], "init_std": spec["init_std"],
            "data_dir": DATA_DIR, "out_dir": os.path.join(OUTPUT_DIR, f"run_{tag}"),
            "tokenizer_path": TOKENIZER_PATH, "test_path": test_path,
            "deadline_unix": deadline, "session_end_unix": SESSION_END, "schedule_hours": None,
            "resume_from": RESUME.get(tag), "train": train, "gen": GEN,
        }
        path = os.path.join(WORK, f"run_{tag}.json")
        json.dump(cfg, open(path, "w"), indent=2)
        return cfg, path


    def launch(tag, cfg_path, gpu):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS="1", PYTHONUNBUFFERED="1")
        log = open(os.path.join(OUTPUT_DIR, f"train_{tag}.log"), "w")
        return subprocess.Popen([sys.executable, "train_lm.py", cfg_path], cwd=WORK, env=env, stdout=log, stderr=subprocess.STDOUT)


    def tail(tag, n=3):
        path = os.path.join(OUTPUT_DIR, f"train_{tag}.log")
        lines = open(path).read().strip().splitlines() if os.path.exists(path) else []
        return lines[-n:]


    def wait_all(procs, every=600):
        while any(p.poll() is None for p in procs.values()):
            time.sleep(every)
            print(time.strftime("%H:%M:%S"), f"| {(time.time() - SESSION_START) / 3600:.2f} h into session")
            for tag in procs:
                for line in tail(tag, 2):
                    print(f"  [{tag}] {line[:300]}")
        for tag, p in procs.items():
            print(f"model {tag} exited with code {p.returncode}")
            if p.returncode:
                print("\n".join(tail(tag, 30)))


    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if N_GPUS >= 2:
        _, path_a = run_config("A", MODEL_A, TRAIN_A, train_deadline)
        _, path_b = run_config("B", MODEL_B, TRAIN_B, train_deadline)
        wait_all({"A": launch("A", path_a, 0), "B": launch("B", path_b, 1)})
    else:                                          # single GPU fallback: A then B, half the time each
        print("Only one GPU visible: training sequentially.")
        mid = time.time() + (train_deadline - time.time()) / 2 - GEN_RESERVE_MIN * 60 / 2
        _, path_a = run_config("A", MODEL_A, TRAIN_A, mid)
        wait_all({"A": launch("A", path_a, 0)})
        _, path_b = run_config("B", MODEL_B, TRAIN_B, train_deadline)
        wait_all({"B": launch("B", path_b, 0)})

## Stage 5 — Codabench zips + curves

In [ ]:
# submissions + curves
if DATA_ONLY:
    print("DATA_ONLY = True -> stage 5 (submissions) skipped")
else:
    import zipfile
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for tag in ("A", "B"):
        run = os.path.join(OUTPUT_DIR, f"run_{tag}")
        pred = os.path.join(run, "prediction.json")
        if os.path.exists(pred):
            preds = json.load(open(pred, encoding="utf-8"))
            assert set(preds) == set(test_data), f"model {tag}: prediction ids differ from test ids"
            n_words = [len(v["text"].split()) for v in preds.values()]
            shutil.copy(pred, os.path.join(OUTPUT_DIR, f"prediction_{tag}.json"))
            with zipfile.ZipFile(os.path.join(OUTPUT_DIR, f"submission_{tag}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
                z.write(pred, arcname="prediction.json")
            print(f"model {tag}: {len(preds)} predictions, words min/mean {min(n_words)}/{np.mean(n_words):.1f} "
                  f"-> submission_{tag}.zip")
        else:
            print(f"model {tag}: no prediction.json (see train_{tag}.log)")
        logf = os.path.join(run, "log.jsonl")
        if os.path.exists(logf):
            recs = [json.loads(l) for l in open(logf)]
            tr = [r for r in recs if r["msg"] == "train"]
            ev = [r for r in recs if r["msg"] in ("eval", "final_eval")]
            axes[0].plot([r["tokens_M"] for r in tr], [r["loss"] for r in tr], label=f"{tag} train", alpha=0.7)
            axes[0].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-", label=f"{tag} val")
            axes[1].plot([r["tokens_M"] for r in tr], [r["tok_per_s"] for r in tr], label=tag)
    axes[0].set(xlabel="tokens (M)", ylabel="loss", title="train / val loss")
    axes[1].set(xlabel="tokens (M)", ylabel="tokens / s", title="throughput per T4")
    for ax in axes:
        ax.grid(True); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
    print(sorted(os.listdir(OUTPUT_DIR)))